# 04 — Fine-tune Dense Retriever với multilingual-E5-base

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from __future__ import annotations

from pathlib import Path
import gc
import json
import random

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
)
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers.training_args import BatchSamplers
from transformers import EarlyStoppingCallback

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB",
    )

/tmp/ipykernel_1969/4204375445.py:13: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import (
/tmp/ipykernel_1969/4204375445.py:19: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import InformationRetrievalEvaluator
/tmp/ipykernel_1969/4204375445.py:20: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import BatchSamplers


PyTorch: 2.11.0+cu128
CUDA: True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 79.25 GB


In [31]:
# SET PATHS AND TRAINING CONFIG

PROJECT_ROOT = Path("/content/drive/MyDrive/UIT_Legal_System")

PROCESSED_DIR = Path("/content/drive/MyDrive/UIT_Legal_System/Dataset/processed")
MINING_DIR = Path("/content/drive/MyDrive/UIT_Legal_System/Dataset/artifacts/hard_negative_mining")

OUTPUT_DIR = Path("/content/drive/MyDrive/UIT_Legal_System/Source Code/Embedding Model/multilingual_e5_base_finetuned_ver2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_GROUPS_PATH = MINING_DIR / "train_groups.parquet"
CORPUS_PATH = PROCESSED_DIR / "corpus.parquet"

BASE_MODEL = "intfloat/multilingual-e5-base"
EVAL_SPLIT = "strict_val"

MAX_QUERY_LENGTH = 128
MAX_PASSAGE_LENGTH = 512

NUM_HARD_NEGATIVES = 8

NUM_EPOCHS = 10
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.10
WEIGHT_DECAY = 0.01

PER_DEVICE_TRAIN_BATCH_SIZE = 16
PER_DEVICE_EVAL_BATCH_SIZE = 64
GRADIENT_ACCUMULATION_STEPS = 2

MINI_BATCH_SIZE = 8

LOGGING_STEPS = 10
SAVE_TOTAL_LIMIT = 2
EARLY_STOPPING_PATIENCE = 2

RESUME_FROM_CHECKPOINT = True

print("Output:", OUTPUT_DIR)

Output: /content/drive/MyDrive/UIT_Legal_System/Source Code/Embedding Model/multilingual_e5_base_finetuned_ver2


In [32]:
# LOAD INPUTS

required_paths = [
    TRAIN_GROUPS_PATH,
    CORPUS_PATH,
    PROCESSED_DIR / f"{EVAL_SPLIT}_queries.json",
    PROCESSED_DIR / f"{EVAL_SPLIT}_qrels.json",
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Thiếu input:\n- " + "\n- ".join(missing_paths)
    )

groups_df = pd.read_parquet(TRAIN_GROUPS_PATH)
corpus_df = pd.read_parquet(CORPUS_PATH)

print("Training groups:", len(groups_df))
print("Corpus passages:", len(corpus_df))
display(groups_df.head(2))

Training groups: 7718
Corpus passages: 1045


,qa_id,query,positive,positive_passage_id,positive_document,positive_article,hard_negatives,num_hard_negatives,model_key,corpus_field
0,qa_0005442ef74b8832ada8,Giảng viên ĐTTX được ưu tiên sử dụng những thi...,Văn bản: QUY CHẾ ĐÀO TẠO TỪ XA TRÌNH ĐỘ ĐẠI HỌ...,passage_0e486363680f96bde9ae,QUY CHẾ ĐÀO TẠO TỪ XA TRÌNH ĐỘ ĐẠI HỌC CỦA TRƯ...,Điều 21. Quyền và trách nhiệm của giảng viên,[{'article': 'Điều 6. Đặt hoặc thôi đặt trạm đ...,12,bge_m3,embedding_text
1,qa_00055f2babca7a709df7,"Quy định nào đặt trách nhiệm cho các khoa, bộ ...",Văn bản: QUY ĐỊNH Về tiêu chuẩn giảng viên giả...,passage_9f00a198fac6b5e27859,QUY ĐỊNH Về tiêu chuẩn giảng viên giảng dạy mô...,Điều 12. Tổ chức thực hiện,"[{'article': 'Điều 8. Xây dựng, thẩm định học ...",12,bge_m3,embedding_text


In [33]:
# E5 INPUT FORMAT

def format_query(question: str) -> str:
    return f"query: {str(question).strip()}"

def format_passage(passage: str) -> str:
    return f"passage: {str(passage).strip()}"

def choose_negative_text(item: dict) -> str:
    return str(
        item.get("text")
        or item.get("content")
        or ""
    ).strip()

print(format_query("Sinh viên được bảo lưu khi nào?"))
print(format_passage("Sinh viên được bảo lưu trong các trường hợp..."))

query: Sinh viên được bảo lưu khi nào?
passage: Sinh viên được bảo lưu trong các trường hợp...


In [34]:
# BUILD TRAIN DATASET

records = []
skipped = 0

for row in groups_df.to_dict(orient="records"):
    negatives = row["hard_negatives"]

    if isinstance(negatives, np.ndarray):
        negatives = negatives.tolist()

    if not isinstance(negatives, list):
        skipped += 1
        continue

    negative_texts = [
        choose_negative_text(item)
        for item in negatives
        if isinstance(item, dict) and choose_negative_text(item)
    ]

    negative_texts = negative_texts[:NUM_HARD_NEGATIVES]

    if len(negative_texts) < NUM_HARD_NEGATIVES:
        skipped += 1
        continue

    record = {
        "anchor": format_query(row["query"]),
        "positive": format_passage(row["positive"]),
    }

    for index, negative_text in enumerate(negative_texts, start=1):
        record[f"negative_{index}"] = format_passage(negative_text)

    records.append(record)

if not records:
    raise RuntimeError("Không tạo được training record.")

train_dataset = Dataset.from_list(records)

print("Usable records:", len(train_dataset))
print("Skipped:", skipped)
print("Columns:", train_dataset.column_names)

Usable records: 7718
Skipped: 0
Columns: ['anchor', 'positive', 'negative_1', 'negative_2', 'negative_3', 'negative_4', 'negative_5', 'negative_6', 'negative_7', 'negative_8']


In [35]:
# BUILD INFORMATION RETRIEVAL EVALUATOR

with (
    PROCESSED_DIR / f"{EVAL_SPLIT}_queries.json"
).open("r", encoding="utf-8") as file:
    eval_queries_raw = json.load(file)

with (
    PROCESSED_DIR / f"{EVAL_SPLIT}_qrels.json"
).open("r", encoding="utf-8") as file:
    eval_qrels_raw = json.load(file)

# Dùng embedding_text để giữ metadata văn bản + điều khoản.
corpus_field = "embedding_text"

eval_corpus = {
    str(row["passage_id"]): format_passage(row[corpus_field])
    for row in corpus_df.to_dict(orient="records")
}

eval_queries = {
    str(query_id): format_query(question)
    for query_id, question in eval_queries_raw.items()
}

eval_relevant_docs = {
    str(query_id): set(map(str, passage_ids))
    for query_id, passage_ids in eval_qrels_raw.items()
}

valid_query_ids = [
    query_id
    for query_id in eval_queries
    if query_id in eval_relevant_docs
    and eval_relevant_docs[query_id].issubset(eval_corpus.keys())
]

eval_queries = {
    query_id: eval_queries[query_id]
    for query_id in valid_query_ids
}

eval_relevant_docs = {
    query_id: eval_relevant_docs[query_id]
    for query_id in valid_query_ids
}

ir_evaluator = InformationRetrievalEvaluator(
    queries=eval_queries,
    corpus=eval_corpus,
    relevant_docs=eval_relevant_docs,
    name=EVAL_SPLIT,
    mrr_at_k=[10],
    ndcg_at_k=[10],
    accuracy_at_k=[1, 3, 5, 10, 20],
    precision_recall_at_k=[1, 3, 5, 10, 20],
    map_at_k=[10],
    batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    show_progress_bar=True,
    write_csv=True,
)

print("Evaluation queries:", len(eval_queries))
print("Evaluation corpus:", len(eval_corpus))

Evaluation queries: 933
Evaluation corpus: 1045


In [36]:
# LOAD BASE E5 MODEL

def clear_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

clear_memory()

device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(
    BASE_MODEL,
    device=device,
    model_kwargs={
        "torch_dtype": torch.bfloat16
        if torch.cuda.is_available()
        else torch.float32,
    },
)

model.max_seq_length = MAX_PASSAGE_LENGTH

print(model)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'XLMRobertaModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)


In [37]:
# BASELINE EVALUATION

baseline_metrics = ir_evaluator(
    model,
    output_path=str(OUTPUT_DIR / "baseline_evaluation"),
)

baseline_metrics = {
    key: float(value)
    for key, value in baseline_metrics.items()
}

with (
    OUTPUT_DIR / "baseline_metrics.json"
).open("w", encoding="utf-8") as file:
    json.dump(
        baseline_metrics,
        file,
        ensure_ascii=False,
        indent=2,
    )

print(json.dumps(baseline_metrics, ensure_ascii=False, indent=2))

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:00<00:00,  1.25it/s]

{
  "strict_val_cosine_accuracy@1": 0.2165058949624866,
  "strict_val_cosine_accuracy@3": 0.4608788853161844,
  "strict_val_cosine_accuracy@5": 0.5862808145766345,
  "strict_val_cosine_accuracy@10": 0.7191854233654876,
  "strict_val_cosine_accuracy@20": 0.8478027867095391,
  "strict_val_cosine_precision@1": 0.2165058949624866,
  "strict_val_cosine_precision@3": 0.15362629510539477,
  "strict_val_cosine_precision@5": 0.11725616291532688,
  "strict_val_cosine_precision@10": 0.07191854233654876,
  "strict_val_cosine_precision@20": 0.04239013933547695,
  "strict_val_cosine_recall@1": 0.2165058949624866,
  "strict_val_cosine_recall@3": 0.4608788853161844,
  "strict_val_cosine_recall@5": 0.5862808145766345,
  "strict_val_cosine_recall@10": 0.7191854233654876,
  "strict_val_cosine_recall@20": 0.8478027867095391,
  "strict_val_cosine_ndcg@10": 0.4519691906872627,
  "strict_val_cosine_mrr@10": 0.36809872573538105,
  "strict_val_cosine_map@10": 0.3680987257353817
}


In [38]:
# LOSS

train_loss = losses.CachedMultipleNegativesRankingLoss(
    model=model,
    scale=20.0,
    mini_batch_size=MINI_BATCH_SIZE,
    show_progress_bar=False,
)

print(train_loss)

CachedMultipleNegativesRankingLoss(
  (model): SentenceTransformer(
    (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'XLMRobertaModel'})
    (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})
    (2): Normalize({})
  )
)


In [39]:
effective_anchor_batch_size = (
    PER_DEVICE_TRAIN_BATCH_SIZE
    * GRADIENT_ACCUMULATION_STEPS
)

training_args = SentenceTransformerTrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),

    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,

    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    bf16=torch.cuda.is_available(),
    fp16=False,
    tf32=False, # Changed from torch.cuda.is_available() to False

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False,
    },

    batch_sampler=BatchSamplers.NO_DUPLICATES,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_strategy="steps",
    logging_steps=LOGGING_STEPS,

    load_best_model_at_end=True,
    metric_for_best_model=f"{EVAL_SPLIT}_cosine_recall@10",
    greater_is_better=True,

    save_total_limit=SAVE_TOTAL_LIMIT,
    report_to=["tensorboard"],

    dataloader_num_workers=2,
    dataloader_pin_memory=True,

    seed=SEED,
    data_seed=SEED,

    remove_unused_columns=False,
)

print("Effective anchor batch size:", effective_anchor_batch_size)
print(training_args)

Effective anchor batch size: 32
SentenceTransformerTrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
batch_sampler=BatchSamplers.NO_DUPLICATES,
bf16=True,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=2,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=False,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_ba

In [40]:
# TRAINER

trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=ir_evaluator,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=0.0,
        )
    ],
)

print("Trainer ready.")

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Trainer ready.


In [41]:
# FIND LATEST CHECKPOINT

def find_latest_checkpoint(checkpoint_root: Path) -> str | None:
    if not checkpoint_root.exists():
        return None

    checkpoints = []

    for path in checkpoint_root.glob("checkpoint-*"):
        try:
            step = int(path.name.split("-")[-1])
            checkpoints.append((step, path))
        except ValueError:
            continue

    if not checkpoints:
        return None

    checkpoints.sort(key=lambda item: item[0])
    return str(checkpoints[-1][1])

latest_checkpoint = (
    find_latest_checkpoint(OUTPUT_DIR / "checkpoints")
    if RESUME_FROM_CHECKPOINT
    else None
)

print("Resume checkpoint:", latest_checkpoint)

Resume checkpoint: /content/drive/MyDrive/UIT_Legal_System/Source Code/Embedding Model/multilingual_e5_base_finetuned_ver2/checkpoints/checkpoint-726


In [42]:
# TRAIN

train_result = trainer.train(
    resume_from_checkpoint=latest_checkpoint
)

trainer.save_state()

train_metrics = {
    key: float(value)
    for key, value in train_result.metrics.items()
}

with (
    OUTPUT_DIR / "train_metrics.json"
).open("w", encoding="utf-8") as file:
    json.dump(
        train_metrics,
        file,
        ensure_ascii=False,
        indent=2,
    )

print(json.dumps(train_metrics, ensure_ascii=False, indent=2))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Strict Val Cosine Accuracy@1,Strict Val Cosine Accuracy@3,Strict Val Cosine Accuracy@5,Strict Val Cosine Accuracy@10,Strict Val Cosine Accuracy@20,Strict Val Cosine Precision@1,Strict Val Cosine Precision@3,Strict Val Cosine Precision@5,Strict Val Cosine Precision@10,Strict Val Cosine Precision@20,Strict Val Cosine Recall@1,Strict Val Cosine Recall@3,Strict Val Cosine Recall@5,Strict Val Cosine Recall@10,Strict Val Cosine Recall@20,Strict Val Cosine Ndcg@10,Strict Val Cosine Mrr@10,Strict Val Cosine Map@10
4,1.109221,No log,0.350482,0.607717,0.704180,0.826367,0.911040,0.350482,0.202572,0.140836,0.082637,0.045552,0.350482,0.607717,0.704180,0.826367,0.911040,0.579332,0.501224,0.501224
5,1.100808,No log,0.361200,0.642015,0.722401,0.840300,0.918542,0.361200,0.214005,0.144480,0.084030,0.045927,0.361200,0.642015,0.722401,0.840300,0.918542,0.597287,0.519967,0.519967
6,1.193521,No log,0.370847,0.646302,0.735263,0.845659,0.919614,0.370847,0.215434,0.147053,0.084566,0.045981,0.370847,0.646302,0.735263,0.845659,0.919614,0.604202,0.527331,0.527331
7,1.116709,No log,0.390139,0.649518,0.734191,0.851018,0.922830,0.390139,0.216506,0.146838,0.085102,0.046141,0.390139,0.649518,0.734191,0.851018,0.922830,0.614167,0.539098,0.539098
8,1.074256,No log,0.404073,0.654877,0.739550,0.851018,0.923901,0.404073,0.218292,0.147910,0.085102,0.046195,0.404073,0.654877,0.739550,0.851018,0.923901,0.621322,0.548543,0.548543
9,1.024257,No log,0.394427,0.654877,0.738478,0.855305,0.921758,0.394427,0.218292,0.147696,0.085531,0.046088,0.394427,0.654877,0.738478,0.855305,0.921758,0.619329,0.544611,0.544611
10,0.915619,No log,0.400857,0.655949,0.737406,0.853162,0.923901,0.400857,0.218650,0.147481,0.085316,0.046195,0.400857,0.655949,0.737406,0.853162,0.923901,0.620514,0.546850,0.546850


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:01<00:00,  1.28s/it]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{
  "train_runtime": 5974.5778,
  "train_samples_per_second": 12.918,
  "train_steps_per_second": 0.405,
  "total_flos": 0.0,
  "train_loss": 0.8096845932243284,
  "epoch": 10.0
}


In [43]:
# FINAL EVALUATION
final_metrics = ir_evaluator(
    model,
    output_path=str(OUTPUT_DIR / "final_evaluation"),
)

final_metrics = {
    key: float(value)
    for key, value in final_metrics.items()
}

with (
    OUTPUT_DIR / "final_metrics.json"
).open("w", encoding="utf-8") as file:
    json.dump(
        final_metrics,
        file,
        ensure_ascii=False,
        indent=2,
    )

comparison_rows = []

for metric_name in sorted(
    set(baseline_metrics) | set(final_metrics)
):
    baseline = float(
        baseline_metrics.get(metric_name, np.nan)
    )
    fine_tuned = float(
        final_metrics.get(metric_name, np.nan)
    )

    comparison_rows.append(
        {
            "metric": metric_name,
            "baseline": baseline,
            "fine_tuned": fine_tuned,
            "absolute_change": fine_tuned - baseline,
        }
    )

comparison_df = pd.DataFrame(comparison_rows)

comparison_df.to_csv(
    OUTPUT_DIR / "baseline_vs_finetuned.csv",
    index=False,
    encoding="utf-8-sig",
)

display(comparison_df)

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:00<00:00,  1.19it/s]


,metric,baseline,fine_tuned,absolute_change
0,strict_val_cosine_accuracy@1,0.216506,0.394427,0.177921
1,strict_val_cosine_accuracy@10,0.719185,0.855305,0.136120
2,strict_val_cosine_accuracy@20,0.847803,0.921758,0.073955
3,strict_val_cosine_accuracy@3,0.460879,0.654877,0.193998
4,strict_val_cosine_accuracy@5,0.586281,0.738478,0.152197
5,strict_val_cosine_map@10,0.368099,0.544611,0.176512
6,strict_val_cosine_mrr@10,0.368099,0.544611,0.176512
7,strict_val_cosine_ndcg@10,0.451969,0.619329,0.167360
8,strict_val_cosine_precision@1,0.216506,0.394427,0.177921
9,strict_val_cosine_precision@10,0.071919,0.085531,0.013612


In [44]:
# SAVE FINAL SENTENCE-TRANSFORMER MODEL

final_model_dir = OUTPUT_DIR / "sentence_transformer"

model.save_pretrained(
    str(final_model_dir)
)

print("Saved final model:", final_model_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved final model: /content/drive/MyDrive/UIT_Legal_System/Source Code/Embedding Model/multilingual_e5_base_finetuned_ver2/sentence_transformer


In [45]:
# RELOAD SMOKE TEST

clear_memory()

reloaded_model = SentenceTransformer(
    str(final_model_dir),
    device="cuda" if torch.cuda.is_available() else "cpu",
)

smoke_query = format_query(
    "Sinh viên bị cảnh báo học tập trong trường hợp nào?"
)

smoke_passage = format_passage(
    "Sinh viên bị cảnh báo học tập khi kết quả học tập "
    "không đạt yêu cầu theo quy định."
)

smoke_embeddings = reloaded_model.encode(
    [smoke_query, smoke_passage],
    normalize_embeddings=True,
    convert_to_numpy=True,
)

smoke_similarity = float(
    smoke_embeddings[0] @ smoke_embeddings[1]
)

print("Embedding shape:", smoke_embeddings.shape)
print("Smoke similarity:", smoke_similarity)

del reloaded_model
clear_memory()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding shape: (2, 768)
Smoke similarity: 0.558239758014679


In [46]:
# SAVE TRAINING CONFIG

training_config = {
    "schema_version": "1.0",
    "model_key": "multilingual_e5_base",
    "base_model": BASE_MODEL,
    "training_type": "full_finetune",
    "loss": "CachedMultipleNegativesRankingLoss",
    "eval_split": EVAL_SPLIT,
    "query_instruction": "",
    "query_format": "query: {question}",
    "passage_format": "passage: {passage}",
    "corpus_field": corpus_field,
    "embedding_dimension": 768,
    "max_query_length": MAX_QUERY_LENGTH,
    "max_passage_length": MAX_PASSAGE_LENGTH,
    "num_hard_negatives": NUM_HARD_NEGATIVES,
    "epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "warmup_ratio": WARMUP_RATIO,
    "weight_decay": WEIGHT_DECAY,
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "effective_anchor_batch_size": effective_anchor_batch_size,
    "cached_loss_mini_batch_size": MINI_BATCH_SIZE,
    "precision": "bf16" if torch.cuda.is_available() else "fp32",
    "output_model": str(final_model_dir),
}

with (
    OUTPUT_DIR / "training_config.json"
).open("w", encoding="utf-8") as file:
    json.dump(
        training_config,
        file,
        ensure_ascii=False,
        indent=2,
    )

print(json.dumps(training_config, ensure_ascii=False, indent=2))

{
  "schema_version": "1.0",
  "model_key": "multilingual_e5_base",
  "base_model": "intfloat/multilingual-e5-base",
  "training_type": "full_finetune",
  "loss": "CachedMultipleNegativesRankingLoss",
  "eval_split": "strict_val",
  "query_instruction": "",
  "query_format": "query: {question}",
  "passage_format": "passage: {passage}",
  "corpus_field": "embedding_text",
  "embedding_dimension": 768,
  "max_query_length": 128,
  "max_passage_length": 512,
  "num_hard_negatives": 8,
  "epochs": 10,
  "learning_rate": 2e-05,
  "warmup_ratio": 0.1,
  "weight_decay": 0.01,
  "per_device_train_batch_size": 16,
  "gradient_accumulation_steps": 2,
  "effective_anchor_batch_size": 32,
  "cached_loss_mini_batch_size": 8,
  "precision": "bf16",
  "output_model": "/content/drive/MyDrive/UIT_Legal_System/Source Code/Embedding Model/multilingual_e5_base_finetuned_ver2/sentence_transformer"
}


In [47]:
# FINAL VALIDATION

required_outputs = [
    OUTPUT_DIR / "baseline_metrics.json",
    OUTPUT_DIR / "train_metrics.json",
    OUTPUT_DIR / "final_metrics.json",
    OUTPUT_DIR / "baseline_vs_finetuned.csv",
    OUTPUT_DIR / "training_config.json",
    final_model_dir,
]

missing_outputs = [
    str(path)
    for path in required_outputs
    if not path.exists()
]

if missing_outputs:
    raise RuntimeError(
        "Thiếu output:\n- "
        + "\n- ".join(missing_outputs)
    )

print("E5 fine-tuning completed successfully.\n")

for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        print(
            f"{path.name:42s} "
            f"{path.stat().st_size / 1024:12.2f} KB"
        )
    else:
        print(f"{path.name:42s} <directory>")

E5 fine-tuning completed successfully.

baseline_evaluation                        <directory>
baseline_metrics.json                              0.95 KB
baseline_vs_finetuned.csv                          1.57 KB
checkpoints                                <directory>
final_evaluation                           <directory>
final_metrics.json                                 0.94 KB
sentence_transformer                       <directory>
train_metrics.json                                 0.17 KB
training_config.json                               0.87 KB
